# IPN variable length — Step 5 windowing smoke test

Audit window 256/stride 128 trên toàn bộ cache Drive, kiểm tra coverage, velocity boundary, model forward và mean-logit aggregation theo clip. Notebook không huấn luyện.

In [ ]:
from pathlib import Path
import datetime as dt
import json
import subprocess
import sys
import torch
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/HuyenHoang28/DynamicHandGestureWithXAI.git'
REPO_BRANCH = 'HuyenHoang'
REPO_ROOT = Path('/content/dynamic_hand_gesture_recognition')
DATA_ROOT = Path('/content/drive/MyDrive/ipn-extract-data')
CACHE_ROOT = DATA_ROOT / 'keypoint_tensor_cache'
MAX_FRAMES, STRIDE = 256, 128
EXPECTED = {'train': 2443, 'val': 589, 'test': 1101}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def run(argv, cwd=None):
    print('$', ' '.join(map(str, argv)))
    subprocess.run([str(value) for value in argv], cwd=cwd, check=True)


In [ ]:
if (REPO_ROOT / '.git').is_dir():
    run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_ROOT)
    run(['git', 'checkout', REPO_BRANCH], cwd=REPO_ROOT)
    run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO_ROOT)
else:
    run(['git', 'clone', '--filter=blob:none', '--branch', REPO_BRANCH, REPO_URL, REPO_ROOT])
sys.path[:0] = [str(REPO_ROOT / 'final/code'), str(REPO_ROOT / 'final/code/neural_only_experiments')]
from neural_only_experiments.variable_dataset import (
    WindowedVariableKeypointDataset, aggregate_window_logits,
    collate_variable_batch, variable_cache_path, window_spans,
)
from src_neurosymbolic.variable_keypoint_cache import load_variable_cache
from models import NeuralOnlyClassifier
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
report_path = CACHE_ROOT / 'variable_v1' / 'STEP5_WINDOW_REPORT.json'
tmp = report_path.with_suffix('.json.tmp')
tmp.write_text(json.dumps({'status': 'RUNNING', 'timestamp_utc': dt.datetime.now(dt.timezone.utc).isoformat(), 'repo_commit': commit}, indent=2) + '\n', encoding='utf-8')
tmp.replace(report_path)
print({'commit': commit, 'device': str(DEVICE)})

In [ ]:
step4 = CACHE_ROOT / 'variable_v1' / 'STEP4_SMOKE_REPORT.json'
assert step4.is_file() and json.loads(step4.read_text())['status'] == 'PASS', 'Bước 4 chưa PASS'
datasets, audit = {}, {}
for split, expected in EXPECTED.items():
    ds = WindowedVariableKeypointDataset(DATA_ROOT / f'{split}.txt', CACHE_ROOT, split, MAX_FRAMES, STRIDE)
    assert len(ds.samples) == expected
    datasets[split] = ds
    by_clip = {}
    for clip_index, window_index, window_count, start, end in ds.windows:
        by_clip.setdefault(clip_index, []).append((window_index, window_count, start, end))
    assert len(by_clip) == expected
    long_clips = 0
    for clip_index, rows in by_clip.items():
        source, _ = ds.samples[clip_index]
        length = load_variable_cache(variable_cache_path(source, CACHE_ROOT))['length']
        expected_spans = window_spans(length, MAX_FRAMES, STRIDE)
        actual_spans = [(start, end) for _, _, start, end in sorted(rows)]
        assert actual_spans == expected_spans
        assert all(count == len(rows) == len(expected_spans) for _, count, _, _ in rows)
        assert [index for index, *_ in sorted(rows)] == list(range(len(rows)))
        assert abs(sum(1 / count for _, count, _, _ in rows) - 1.0) < 1e-9
        covered = set()
        for start, end in actual_spans:
            assert 0 <= start < end <= length and end - start <= MAX_FRAMES
            covered.update(range(start, end))
        assert covered == set(range(length)), f'Window không phủ đủ frame: {source}'
        assert actual_spans[-1][1] == length
        long_clips += int(length > MAX_FRAMES)
    audit[split] = {'clips': expected, 'windows': len(ds), 'long_clips': long_clips}
    assert audit[split]['windows'] >= expected
    print(split, audit[split])

In [ ]:
from torch.utils.data import DataLoader, Subset
train_ds = datasets['train']
long_window_ids = [i for i, row in enumerate(train_ds.windows) if row[2] > 1]
assert long_window_ids, 'Không tìm thấy window của clip dài'
chosen_clip = train_ds.windows[long_window_ids[0]][0]
chosen_ids = [i for i, row in enumerate(train_ds.windows) if row[0] == chosen_clip]
items = [train_ds[i] for i in chosen_ids]
assert all(torch.count_nonzero(item['velocity'][0]).item() == 0 for item in items)
assert abs(sum(item['sample_weight'] for item in items) - 1.0) < 1e-9
loader = DataLoader(Subset(train_ds, chosen_ids), batch_size=len(chosen_ids), shuffle=False, collate_fn=collate_variable_batch)
batch = next(iter(loader))
model = NeuralOnlyClassifier(backbone='transformer', num_classes=13, num_frames=256, d_model=192, layers=4, heads=8, dropout=0.0, use_motion_features=True, pooling='masked_mean').to(DEVICE).eval()
with torch.no_grad():
    logits = model(batch['keypoints'].to(DEVICE), padding_mask=batch['padding_mask'].to(DEVICE), velocity=batch['velocity'].to(DEVICE))
assert torch.isfinite(logits).all()
aggregated = aggregate_window_logits(logits.cpu(), batch)
assert aggregated['logits'].shape == (1, 13) and aggregated['labels'].numel() == 1
print({'chosen_clip_windows': len(chosen_ids), 'window_batch': list(batch['keypoints'].shape), 'clip_logits': list(aggregated['logits'].shape)})

In [ ]:
all_logits, metadata = [], {key: [] for key in ('label','source','split','clip_index','window_index','window_count')}
for split, ds in datasets.items():
    for clip_index, window_index, window_count, start, end in ds.windows:
        source, label = ds.samples[clip_index]
        row = torch.zeros(13)
        row[label] = 1.0
        all_logits.append(row)
        for key, value in (('label',label),('source',source),('split',split),('clip_index',clip_index),('window_index',window_index),('window_count',window_count)):
            metadata[key].append(value)
clip_output = aggregate_window_logits(torch.stack(all_logits), metadata)
assert clip_output['logits'].shape[0] == sum(EXPECTED.values())
assert torch.equal(clip_output['logits'].argmax(1), clip_output['labels'])
report = {'status':'PASS','timestamp_utc':dt.datetime.now(dt.timezone.utc).isoformat(),'repo_commit':commit,'max_frames':MAX_FRAMES,'stride':STRIDE,'splits':audit,'total_clip_predictions':int(clip_output['labels'].numel()),'aggregation':'mean_logits'}
tmp.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
tmp.replace(report_path)
print(json.dumps(report, indent=2))
print('Saved:', report_path)